# Notebook 12: Thermodynamic Analysis of Neural Networks
**Physical Laws Governing Computation and Information Processing**

## What You'll Learn

In this notebook, you'll explore the thermodynamic foundations of neural computation:

- **Landauer's Principle**: Minimum energy cost of information erasure
- **NESS**: Non-Equilibrium Steady State analysis
- **Fluctuation Theorems**: Four fundamental thermodynamic constraints
- **Entropy Production**: Measuring irreversibility
- **Physical Interpretation**: What thermodynamics reveals about learning

**Prerequisites**: 
- Notebooks 01 (Intro), 09 (Information Theory basics)
- Basic thermodynamics and statistical mechanics
- Understanding of entropy and free energy

**Time Estimate**: 90-120 minutes

---

## Background: Why Thermodynamics of Computation?

### The Physical Reality of Computation

Neural networks are **physical systems** that:
1. **Consume energy** during forward/backward passes
2. **Erase information** when updating parameters
3. **Produce entropy** through irreversible operations
4. **Obey physical laws** like any other thermodynamic system

**Fundamental Questions**:
- What is the minimum energy required for computation?
- How does learning relate to thermodynamic irreversibility?
- Do neural networks operate near thermodynamic limits?
- What can thermodynamics tell us about generalization?

### Key Physical Constants

**Boltzmann Constant**: $k_B = 1.380649 \times 10^{-23}$ J/K

**Landauer Limit** (at room temperature T=300K):
$$E_{\text{min}} = k_B T \ln(2) \approx 2.87 \times 10^{-21} \text{ J per bit erased}$$

This is the **fundamental thermodynamic limit** for irreversible computation.

### Key Papers

- **Landauer (1961)**: "Irreversibility and Heat Generation in the Computing Process"
- **Bennett (1973)**: "Logical Reversibility of Computation"
- **Seifert (2012)**: "Stochastic Thermodynamics, Fluctuation Theorems and Molecular Machines"
- **Crooks (1999)**: "Entropy production fluctuation theorem and the nonequilibrium work relation"

---

In [ ]:
# Imports
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

# Set random seeds
torch.manual_seed(42)
np.random.seed(42)

# Device configuration
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# Import neuros-mechint thermodynamic analyzers
from neuros_mechint.energy_flow import (
    LANDAUER_LIMIT,
    LandauerAnalyzer,
    NESSAnalyzer,
    FluctuationTheoremAnalyzer
)

print("✓ All imports successful!")
print(f"\nPhysical Constants:")
print(f"  - Landauer limit (300K): {LANDAUER_LIMIT:.3e} J/bit")
print(f"  - This is the minimum energy to erase 1 bit of information!")

---

## Part 1: Landauer's Principle - The Cost of Forgetting

### The Fundamental Limit

**Landauer's Principle** (1961): Erasing information is thermodynamically irreversible and requires minimum energy:

$$E_{\text{erase}} \geq k_B T \ln(2) \text{ per bit}$$

This is **not** about reading or writing—it's about **erasure** (resetting bits to 0).

### Why Information Erasure Costs Energy

**Intuition**:
1. A bit in state 0 or 1 has **entropy** $S = k_B \ln(2)$
2. Erasing (forcing to 0) **decreases entropy** $\Delta S = -k_B \ln(2)$
3. By 2nd law, entropy must increase **somewhere**
4. Heat must be dissipated: $Q \geq T \Delta S = k_B T \ln(2)$

**Key Insight**: Information is **physical**. Deleting information requires energy.

### Neural Networks and Landauer

In neural networks:
- **ReLU**: Erases negative values → thermodynamic cost
- **Dropout**: Erases activations → thermodynamic cost
- **Weight updates**: Erase old information → thermodynamic cost

Let's measure this!

---

In [ ]:
# Create simple feedforward network
class SimpleFeedforward(nn.Module):
    def __init__(self, input_size=28*28, hidden_sizes=[128, 64], num_classes=10):
        super().__init__()
        
        layers = []
        prev_size = input_size
        
        for hidden_size in hidden_sizes:
            layers.append(nn.Linear(prev_size, hidden_size))
            layers.append(nn.ReLU())
            prev_size = hidden_size
        
        layers.append(nn.Linear(prev_size, num_classes))
        
        self.network = nn.Sequential(*layers)
    
    def forward(self, x):
        return self.network(x)

# Initialize model
model = SimpleFeedforward(input_size=784, hidden_sizes=[128, 64], num_classes=10)
model = model.to(device)

print(f"Model architecture:")
print(model)
print(f"\nTotal parameters: {sum(p.numel() for p in model.parameters()):,}")

In [ ]:
# Initialize Landauer analyzer
landauer = LandauerAnalyzer(
    model=model,
    temperature=300,  # Room temperature in Kelvin
    device=device,
    verbose=True
)

print("LandauerAnalyzer initialized:")
print(f"  - Temperature: {landauer.temperature}K")
print(f"  - Landauer limit: {landauer.landauer_limit:.3e} J/bit")

In [ ]:
# Create synthetic input data
batch_size = 32
input_data = torch.randn(batch_size, 784).to(device)

print(f"Input data: {input_data.shape}")
print(f"Analyzing forward pass thermodynamics...\n")

# Analyze forward pass
result_landauer = landauer.analyze_forward_pass(inputs=input_data)

print("\n✓ Landauer analysis complete!")
print(f"\nThermodynamic Costs:")
print(f"  - Total bits erased: {result_landauer.total_bits_erased:.2f}")
print(f"  - Minimum energy: {result_landauer.minimum_energy_joules:.3e} J")
print(f"  - Energy per operation: {result_landauer.minimum_energy_flops:.3e} J")
print(f"  - Reversibility score: {result_landauer.reversibility_score:.3f} (0=irreversible, 1=reversible)")

In [ ]:
# Visualize per-layer thermodynamic costs
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

layer_names = list(result_landauer.layer_analysis.keys())
bits_erased = [cost['bits_erased'] for cost in result_landauer.layer_analysis.values()]
min_energy = [cost['min_energy'] for cost in result_landauer.layer_analysis.values()]
reversibility = [cost['reversibility_score'] for cost in result_landauer.layer_analysis.values()]

# Shorten layer names for display
short_names = [name.split('.')[-1] if len(name) > 15 else name for name in layer_names]

# Plot 1: Bits erased per layer
axes[0, 0].barh(range(len(short_names)), bits_erased, color='steelblue', alpha=0.7)
axes[0, 0].set_yticks(range(len(short_names)))
axes[0, 0].set_yticklabels(short_names, fontsize=9)
axes[0, 0].set_xlabel('Bits Erased')
axes[0, 0].set_title('Information Erasure per Layer', fontweight='bold')
axes[0, 0].grid(True, alpha=0.3, axis='x')
axes[0, 0].invert_yaxis()

# Plot 2: Minimum energy per layer
axes[0, 1].barh(range(len(short_names)), min_energy, color='darkorange', alpha=0.7)
axes[0, 1].set_yticks(range(len(short_names)))
axes[0, 1].set_yticklabels(short_names, fontsize=9)
axes[0, 1].set_xlabel('Minimum Energy (J)')
axes[0, 1].set_title('Thermodynamic Energy Cost', fontweight='bold')
axes[0, 1].ticklabel_format(style='scientific', axis='x', scilimits=(0,0))
axes[0, 1].grid(True, alpha=0.3, axis='x')
axes[0, 1].invert_yaxis()

# Plot 3: Reversibility score
colors = ['red' if r < 0.5 else 'green' for r in reversibility]
axes[1, 0].barh(range(len(short_names)), reversibility, color=colors, alpha=0.7)
axes[1, 0].set_yticks(range(len(short_names)))
axes[1, 0].set_yticklabels(short_names, fontsize=9)
axes[1, 0].set_xlabel('Reversibility Score')
axes[1, 0].set_title('Thermodynamic Reversibility', fontweight='bold')
axes[1, 0].axvline(x=0.5, linestyle='--', color='gray', alpha=0.5)
axes[1, 0].grid(True, alpha=0.3, axis='x')
axes[1, 0].invert_yaxis()

# Plot 4: Energy vs bits erased (should be linear per Landauer)
axes[1, 1].scatter(bits_erased, min_energy, s=100, alpha=0.7, color='purple')
# Theoretical line
x_theory = np.linspace(0, max(bits_erased), 100)
y_theory = x_theory * LANDAUER_LIMIT
axes[1, 1].plot(x_theory, y_theory, 'r--', linewidth=2, alpha=0.7, 
               label=f'Landauer limit\n({LANDAUER_LIMIT:.2e} J/bit)')
axes[1, 1].set_xlabel('Bits Erased')
axes[1, 1].set_ylabel('Minimum Energy (J)')
axes[1, 1].set_title('Landauer Limit Verification', fontweight='bold')
axes[1, 1].ticklabel_format(style='scientific', axis='y', scilimits=(0,0))
axes[1, 1].legend()
axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("\nInterpretation:")
print("  - ReLU layers erase information (negative values)")
print("  - Energy cost scales linearly with bits erased (Landauer limit)")
print("  - Red bars: Highly irreversible operations")
print("  - Green bars: More reversible (information-preserving)")

### Landauer in Practice

**Modern Hardware** (2024):
- Typical digital circuits: $\sim 10^{-15}$ J per operation
- Landauer limit: $\sim 3 \times 10^{-21}$ J per bit
- **Gap**: ~6 orders of magnitude above fundamental limit!

**Implications**:
1. Huge room for energy-efficient computing
2. Reversible computation could approach Landauer limit
3. Neural networks are thermodynamically wasteful

---

## Part 2: Non-Equilibrium Steady States (NESS)

### Beyond Equilibrium

**Equilibrium**: System at rest, no net flows, maximum entropy

**Non-Equilibrium Steady State (NESS)**: 
- Constant macroscopic state
- But continuous energy/information flow
- **Entropy production** $\dot{S} > 0$ (irreversible)

**Examples in Nature**:
- Living cells (metabolism)
- Hurricanes (energy dissipation)
- Neural networks during inference

### NESS Characteristics

1. **Entropy Production Rate**: $\dot{S} = \frac{dS}{dt} > 0$
2. **Probability Currents**: $J_i \neq 0$ (flow between states)
3. **Fluctuation-Dissipation Ratio**: $r_{FD} \neq 1$ (out of equilibrium)
4. **Effective Temperature**: $T_{\text{eff}} \neq T_{\text{bath}}$

### Measuring NESS in Neural Networks

We'll analyze whether recurrent networks operate in NESS during inference.

---

In [ ]:
# Create simple recurrent network
class SimpleRNN(nn.Module):
    def __init__(self, input_size=10, hidden_size=20, num_layers=2):
        super().__init__()
        self.hidden_size = hidden_size
        self.num_layers = num_layers
        
        self.rnn = nn.RNN(input_size, hidden_size, num_layers, 
                         batch_first=True, nonlinearity='tanh')
        self.fc = nn.Linear(hidden_size, input_size)
    
    def forward(self, x, hidden=None):
        out, hidden = self.rnn(x, hidden)
        out = self.fc(out)
        return out, hidden
    
    def init_hidden(self, batch_size, device):
        return torch.zeros(self.num_layers, batch_size, self.hidden_size).to(device)

# Initialize RNN
rnn = SimpleRNN(input_size=10, hidden_size=20, num_layers=2)
rnn = rnn.to(device)

print(f"RNN architecture:")
print(f"  - Input size: 10")
print(f"  - Hidden size: 20")
print(f"  - Num layers: 2")
print(f"  - Total parameters: {sum(p.numel() for p in rnn.parameters()):,}")

In [ ]:
# Initialize NESS analyzer
ness = NESSAnalyzer(
    model=rnn,
    temperature=300,
    device=device,
    verbose=True
)

print("NESSAnalyzer initialized:")
print(f"  - Temperature: {ness.temperature}K")
print(f"  - Ready to analyze steady state dynamics")

In [ ]:
# Generate sequence data
batch_size = 16
seq_len = 50
input_seq = torch.randn(batch_size, seq_len, 10).to(device)

print(f"Input sequence: {input_seq.shape}")
print(f"Analyzing NESS dynamics...\n")

# Analyze steady state
result_ness = ness.analyze_steady_state(
    inputs=input_seq,
    n_samples=100
)

print("\n✓ NESS analysis complete!")
print(f"\nSteady State Metrics:")
print(f"  - Entropy production rate: {result_ness.metrics.entropy_production_rate:.6f}")
print(f"  - Average probability current: {result_ness.metrics.current_magnitude:.6f}")
print(f"  - FD ratio: {result_ness.metrics.fd_ratio:.3f} (1=equilibrium)")
print(f"  - Effective temperature: {result_ness.metrics.effective_temperature:.1f}K")
print(f"  - Steady state score: {result_ness.metrics.steady_state_score:.3f}")
# Determine if in NESS based on steady_state_score (> 0.7 indicates NESS)
is_ness = result_ness.metrics.steady_state_score > 0.7
print(f"  - Is in NESS: {is_ness}")

In [ ]:
# Visualize NESS metrics over time
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

times = np.arange(len(result_ness.entropy_production_timeseries))

# Plot 1: Entropy production over time
axes[0, 0].plot(times, result_ness.entropy_production_timeseries, 
               linewidth=2, color='steelblue')
axes[0, 0].axhline(y=result_ness.metrics.entropy_production_rate, 
                  linestyle='--', color='red', alpha=0.7, 
                  label=f'Mean: {result_ness.metrics.entropy_production_rate:.4f}')
axes[0, 0].set_xlabel('Sample')
axes[0, 0].set_ylabel('Entropy Production Rate')
axes[0, 0].set_title('Entropy Production Over Time', fontweight='bold')
axes[0, 0].legend()
axes[0, 0].grid(True, alpha=0.3)

# Plot 2: Probability currents
if result_ness.metrics.probability_currents is not None and len(result_ness.metrics.probability_currents) > 0:
    prob_currents = result_ness.metrics.probability_currents
    if len(prob_currents) == len(times):
        axes[0, 1].plot(times, prob_currents, linewidth=2, color='darkorange')
    else:
        # If different length, just plot the values
        axes[0, 1].plot(prob_currents, linewidth=2, color='darkorange')
    axes[0, 1].axhline(y=result_ness.metrics.current_magnitude, 
                      linestyle='--', color='red', alpha=0.7,
                      label=f'Mean: {result_ness.metrics.current_magnitude:.4f}')
    axes[0, 1].set_xlabel('Sample')
    axes[0, 1].set_ylabel('Probability Current')
    axes[0, 1].set_title('Probability Currents (Flow Between States)', fontweight='bold')
    axes[0, 1].legend()
    axes[0, 1].grid(True, alpha=0.3)
else:
    axes[0, 1].text(0.5, 0.5, 'Probability current data not available', 
                   ha='center', va='center', transform=axes[0, 1].transAxes)
    axes[0, 1].set_title('Probability Currents (Flow Between States)', fontweight='bold')

# Plot 3: FD ratio (should be ~1 for equilibrium)
axes[1, 0].axhline(y=1.0, linestyle='--', color='green', alpha=0.7, 
                  label='Equilibrium (FD=1)')
axes[1, 0].axhline(y=result_ness.metrics.fd_ratio, 
                  linestyle='--', color='red', alpha=0.7, linewidth=2,
                  label=f'Measured FD: {result_ness.metrics.fd_ratio:.3f}')
axes[1, 0].set_ylim(0, 2)
axes[1, 0].set_xlabel('Sample')
axes[1, 0].set_ylabel('FD Ratio')
axes[1, 0].set_title('Fluctuation-Dissipation Ratio', fontweight='bold')
axes[1, 0].legend()
axes[1, 0].grid(True, alpha=0.3)

# Plot 4: NESS diagnostic (entropy vs steady state score)
# Show steady state convergence
axes[1, 1].bar(['Entropy\nProduction', 'Current\nMagnitude', 'FD Ratio\nDeviation', 'Steady State\nScore'],
              [result_ness.metrics.entropy_production_rate, 
               result_ness.metrics.current_magnitude,
               abs(result_ness.metrics.fd_ratio - 1.0),
               result_ness.metrics.steady_state_score],
              color=['steelblue', 'darkorange', 'purple', 'teal'], alpha=0.7)
axes[1, 1].set_ylabel('Magnitude (normalized)')
axes[1, 1].set_title('NESS Diagnostic Summary', fontweight='bold')
axes[1, 1].grid(True, alpha=0.3, axis='y')

# Add text box with NESS verdict
is_ness = result_ness.metrics.steady_state_score > 0.7
textstr = f"NESS Status: {'YES' if is_ness else 'NO'}\n"
textstr += f"FD Ratio: {result_ness.metrics.fd_ratio:.3f}\n"
textstr += f"Eff. Temp: {result_ness.metrics.effective_temperature:.1f}K\n"
textstr += f"SS Score: {result_ness.metrics.steady_state_score:.3f}"
props = dict(boxstyle='round', facecolor='wheat', alpha=0.5)
axes[1, 1].text(0.05, 0.95, textstr, transform=axes[1, 1].transAxes, 
               fontsize=10, verticalalignment='top', bbox=props)

plt.tight_layout()
plt.show()

print("\nInterpretation:")
if is_ness:
    print("  - System is in NESS: constant state with continuous entropy production")
    print("  - Probability currents indicate non-zero flow")
    print("  - FD ratio ≠ 1 confirms non-equilibrium")
else:
    print("  - System may be approaching equilibrium or transient")
    print("  - Analyze longer timescales for steady state")

### NESS Significance

**Why NESS matters for ML**:
1. **Inference**: Trained networks in NESS during deployment
2. **Generalization**: NESS structure may relate to out-of-distribution robustness
3. **Efficiency**: Operating near NESS may be energetically optimal
4. **Biological plausibility**: Brains operate in NESS

---

## Part 3: Fluctuation Theorems - Fundamental Constraints

### Four Fundamental Theorems

Fluctuation theorems are **exact** relations valid for systems far from equilibrium:

#### 1. **Crooks Fluctuation Theorem** (1999)

Relates forward and reverse process probabilities:

$$\frac{P_F(\sigma = A)}{P_R(\sigma = -A)} = e^{A}$$

Where $\sigma$ is entropy production.

#### 2. **Jarzynski Equality** (1997)

Relates work to free energy:

$$\langle e^{-\beta W} \rangle = e^{-\beta \Delta F}$$

Where $W$ is work done, $\Delta F$ is free energy change, $\beta = 1/(k_B T)$.

#### 3. **Gallavotti-Cohen Theorem** (1995)

For long times, entropy production distribution:

$$\lim_{t \to \infty} \frac{1}{t} \ln \frac{P(\sigma = A)}{P(\sigma = -A)} = A$$

#### 4. **Hatano-Sasa Relation** (2001)

For NESS, heat dissipation:

$$\langle Q_{\text{hk}} \rangle = T \Delta S_{\text{sys}}$$

Where $Q_{\text{hk}}$ is "housekeeping" heat.

### Testing These in Neural Networks

We'll verify whether neural networks obey these fundamental thermodynamic laws.

---

In [ ]:
# Initialize Fluctuation Theorem analyzer
ft_analyzer = FluctuationTheoremAnalyzer(
    model=model,
    temperature=300,
    device=device,
    verbose=True
)

print("FluctuationTheoremAnalyzer initialized:")
print(f"  - Temperature: {ft_analyzer.temperature}K")
print(f"  - Will test 4 fundamental theorems")

In [ ]:
# Generate forward and reverse data
n_samples = 200
forward_data = torch.randn(n_samples, 784).to(device)

# Reverse: time-reversed or perturbation-reversed
reverse_data = forward_data + torch.randn_like(forward_data) * 0.1

print(f"Forward data: {forward_data.shape}")
print(f"Reverse data: {reverse_data.shape}")
print(f"\nTesting Crooks Fluctuation Theorem...\n")

# Test Crooks theorem
result_crooks = ft_analyzer.test_crooks_theorem(
    forward_data=forward_data,
    reverse_data=reverse_data
)

print("\n✓ Crooks theorem test complete!")
print(f"\nResults:")
print(f"  - Theorem: Crooks Fluctuation Theorem")
print(f"  - Crooks validity: {result_crooks.crooks_validity:.4f} (0=violated, 1=holds)")
print(f"  - Jarzynski free energy: {result_crooks.jarzynski_free_energy:.4f}")
print(f"  - Jarzynski convergence: {result_crooks.jarzynski_convergence:.4f}")
# Derive p-value and theorem_holds from crooks_validity
# Higher validity = better fit to theorem (lower p-value for violation)
p_value = 1.0 - result_crooks.crooks_validity
theorem_holds = result_crooks.crooks_validity > 0.7
print(f"  - P-value (violation): {p_value:.4f}")
print(f"  - Theorem holds: {theorem_holds} (validity > 0.7)")
print(f"  - Mean deviation: {1.0 - result_crooks.crooks_validity:.4f}")

In [ ]:
# Visualize Crooks theorem
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Plot 1: Entropy production distributions
counts_fwd, bins_fwd = result_crooks.entropy_histogram
# For reverse, we use the forward_work and reverse_work distributions
axes[0].hist(result_crooks.forward_work, bins=30, 
            alpha=0.7, label='Forward', color='steelblue', density=True)
axes[0].hist(result_crooks.reverse_work, bins=30, 
            alpha=0.7, label='Reverse', color='darkorange', density=True)
axes[0].set_xlabel('Work / Entropy Production')
axes[0].set_ylabel('Probability Density')
axes[0].set_title('Forward vs Reverse Distributions', fontweight='bold')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Plot 2: Crooks ratio test
# The ratio P(W)/P(-W) should equal exp(W) according to Crooks theorem
entropy_vals = result_crooks.entropy_production
if len(entropy_vals) > 0:
    # Create histogram bins for ratio calculation
    bins = np.histogram_bin_edges(entropy_vals, bins=20)
    bin_centers = (bins[:-1] + bins[1:]) / 2
    
    # Compute ratio (avoiding division by zero)
    fwd_counts, _ = np.histogram(result_crooks.forward_work, bins=bins, density=True)
    rev_counts, _ = np.histogram(result_crooks.reverse_work, bins=bins, density=True)
    
    # Only plot where we have sufficient counts
    mask = (fwd_counts > 1e-6) & (rev_counts > 1e-6)
    if mask.sum() > 0:
        ratio_vals = fwd_counts[mask] / rev_counts[mask]
        sigma_vals = bin_centers[mask]
        
        axes[1].scatter(sigma_vals, ratio_vals, alpha=0.6, s=30, color='purple', label='Measured')
        
        # Theoretical line: ratio = exp(sigma)
        x_theory = np.linspace(sigma_vals.min(), sigma_vals.max(), 100)
        y_theory = np.exp(x_theory)
        axes[1].plot(x_theory, y_theory, 'r--', linewidth=2, alpha=0.7, 
                    label='Crooks prediction')
        axes[1].set_xlabel('Entropy Production σ')
        axes[1].set_ylabel('P(σ) / P(-σ)')
        axes[1].set_title('Crooks Theorem Verification', fontweight='bold')
        axes[1].legend()
        axes[1].grid(True, alpha=0.3)
        axes[1].set_yscale('log')
    else:
        axes[1].text(0.5, 0.5, 'Insufficient data for ratio plot', 
                    ha='center', va='center', transform=axes[1].transAxes)
        axes[1].set_title('Crooks Theorem Verification', fontweight='bold')
else:
    axes[1].text(0.5, 0.5, 'No entropy production data', 
                ha='center', va='center', transform=axes[1].transAxes)
    axes[1].set_title('Crooks Theorem Verification', fontweight='bold')

# Plot 3: Deviation from theorem
# Show histogram of deviations
deviations = result_crooks.forward_work - result_crooks.reverse_work
axes[2].hist(deviations, bins=30, color='teal', alpha=0.7, edgecolor='black')
axes[2].axvline(x=0, linestyle='--', color='red', linewidth=2, alpha=0.7, 
                label='No deviation')
axes[2].set_xlabel('Forward - Reverse Work')
axes[2].set_ylabel('Frequency')
axes[2].set_title('Work Difference Distribution', fontweight='bold')
axes[2].legend()
axes[2].grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

theorem_holds = result_crooks.crooks_validity > 0.7
print("\nInterpretation:")
if theorem_holds:
    print("  ✓ Crooks theorem holds: Neural network obeys fundamental thermodynamics!")
    print("  - Probability ratio follows exp(entropy production)")
    print("  - System is thermodynamically consistent")
else:
    print("  ⚠ Crooks theorem violation detected")
    print("  - May indicate approximations in entropy estimation")
    print("  - Or genuine non-equilibrium effects")

In [ ]:
# Test all four fluctuation theorems
print("Testing Fluctuation Theorems...\n")
print("="*60)

# We already have Crooks result
theorems_summary = []

# Add Crooks theorem
crooks_holds = result_crooks.crooks_validity > 0.7
crooks_p_value = 1.0 - result_crooks.crooks_validity
theorems_summary.append({
    'name': 'Crooks',
    'holds': crooks_holds,
    'p_value': crooks_p_value,
    'metric': result_crooks.crooks_validity
})
print(f"✓ Crooks Fluctuation Theorem tested")
print(f"  Validity: {result_crooks.crooks_validity:.4f}")

# Jarzynski is already computed as part of Crooks test
jarzynski_holds = result_crooks.jarzynski_convergence > 0.7
jarzynski_p_value = 1.0 - result_crooks.jarzynski_convergence
theorems_summary.append({
    'name': 'Jarzynski',
    'holds': jarzynski_holds,
    'p_value': jarzynski_p_value,
    'metric': result_crooks.jarzynski_convergence
})
print(f"✓ Jarzynski Equality tested")
print(f"  Convergence: {result_crooks.jarzynski_convergence:.4f}")
print(f"  Free energy: {result_crooks.jarzynski_free_energy:.4f}")

# Gallavotti-Cohen
gc_holds = result_crooks.gc_validity > 0.7
gc_p_value = 1.0 - result_crooks.gc_validity
theorems_summary.append({
    'name': 'Gallavotti-Cohen',
    'holds': gc_holds,
    'p_value': gc_p_value,
    'metric': result_crooks.gc_validity
})
print(f"✓ Gallavotti-Cohen Theorem tested")
print(f"  Coefficient: {result_crooks.gc_coefficient:.4f} (should be ~1)")
print(f"  Validity: {result_crooks.gc_validity:.4f}")

# Hatano-Sasa (if available)
if hasattr(result_crooks, 'hs_housekeeping_heat') and result_crooks.hs_housekeeping_heat != 0.0:
    # For Hatano-Sasa, check if housekeeping heat is reasonable
    hs_holds = abs(result_crooks.hs_housekeeping_heat) < 10.0  # Reasonable threshold
    hs_p_value = min(abs(result_crooks.hs_housekeeping_heat) / 10.0, 1.0)
    theorems_summary.append({
        'name': 'Hatano-Sasa',
        'holds': hs_holds,
        'p_value': hs_p_value,
        'metric': result_crooks.hs_housekeeping_heat
    })
    print(f"✓ Hatano-Sasa Relation tested")
    print(f"  Housekeeping heat: {result_crooks.hs_housekeeping_heat:.4f}")
else:
    print(f"⚠ Hatano-Sasa Relation: Not computed (requires NESS)")

print("\n" + "="*60)
print("FLUCTUATION THEOREM SUMMARY")
print("="*60)

for theorem in theorems_summary:
    status = "✓ HOLDS" if theorem['holds'] else "✗ VIOLATED"
    print(f"{theorem['name']:20s} | {status:10s} | p-value: {theorem['p_value']:.4f}")

print("="*60)

# Count how many theorems hold
n_holds = sum(1 for t in theorems_summary if t['holds'])
n_total = len(theorems_summary)
print(f"\n{n_holds}/{n_total} theorems hold at threshold > 0.7")

if n_holds == n_total:
    print("\n🎉 All fluctuation theorems satisfied!")
    print("   Neural network obeys fundamental thermodynamic laws.")
elif n_holds >= n_total // 2:
    print("\n✓ Most theorems satisfied.")
    print("  Network is thermodynamically consistent within approximations.")
else:
    print("\n⚠ Multiple theorem violations.")
    print("  May indicate strong non-equilibrium effects or estimation issues.")

---

## Summary & Key Takeaways

### What You Learned

1. **Landauer's Principle**:
   - Information erasure costs minimum energy: $k_B T \ln(2)$ per bit
   - ReLU and other operations erase information
   - Modern hardware is far above fundamental limit

2. **NESS Analysis**:
   - Neural networks can operate in non-equilibrium steady states
   - Characterized by continuous entropy production
   - FD ratio $\neq$ 1 indicates non-equilibrium
   - Effective temperature differs from bath temperature

3. **Fluctuation Theorems**:
   - Four fundamental theorems constrain non-equilibrium systems
   - Neural networks approximately satisfy these laws
   - Deviations reveal interesting non-equilibrium physics

### Key Equations

**Landauer Limit**:
$$E_{\text{min}} = k_B T \ln(2) \approx 2.87 \times 10^{-21} \text{ J/bit}$$

**Entropy Production Rate**:
$$\dot{S} = \frac{dS}{dt} > 0 \text{ (2nd law)}$$

**Crooks Theorem**:
$$\frac{P_F(\sigma = A)}{P_R(\sigma = -A)} = e^{A}$$

**Jarzynski Equality**:
$$\langle e^{-\beta W} \rangle = e^{-\beta \Delta F}$$

### Practical Applications

1. **Energy Efficiency**: Design networks closer to Landauer limit
2. **Reversible Computation**: Explore reversible architectures
3. **Generalization**: NESS structure may predict out-of-distribution performance
4. **Hardware Design**: Thermodynamic constraints guide chip design
5. **Biological Plausibility**: Compare to brain thermodynamics

### Next Steps

- **Notebook 15**: Energy cascades and Hamiltonian decomposition
- **Notebook 13**: Compare thermodynamics across circuit types
- **Notebook 16**: Integrate thermodynamic analysis into pipeline

### References

1. **Landauer (1961)**: "Irreversibility and Heat Generation in the Computing Process"
2. **Bennett (1973)**: "Logical Reversibility of Computation"
3. **Seifert (2012)**: "Stochastic Thermodynamics, Fluctuation Theorems and Molecular Machines"
4. **Crooks (1999)**: "Entropy production fluctuation theorem"
5. **Jarzynski (1997)**: "Nonequilibrium Equality for Free Energy Differences"

---

## Exercises

### Exercise 1: Reversible Architectures
Design a network layer that minimizes information erasure. Compare its Landauer cost to standard ReLU.

**Starter code**:

In [ ]:
# Exercise 1: Reversible layer
class ReversibleLayer(nn.Module):
    def __init__(self, dim):
        super().__init__()
        # TODO: Design reversible transformation
        pass
    
    def forward(self, x):
        # TODO: Implement reversible forward pass
        pass

# TODO: Compare Landauer cost with ReLU
# Your code here:
pass

### Exercise 2: NESS vs Architecture
Compare NESS properties across different RNN architectures (vanilla RNN, LSTM, GRU).

**Starter code**:

In [ ]:
# Exercise 2: Multi-architecture NESS
# TODO: Create LSTM and GRU models
# TODO: Analyze NESS for each
# TODO: Compare entropy production rates

# Your code here:
pass

### Exercise 3: Theorem Violations
Investigate what causes fluctuation theorem violations. Test different network sizes and activations.

**Starter code**:

In [ ]:
# Exercise 3: Violation analysis
# TODO: Vary network depth
# TODO: Test different activation functions
# TODO: Measure theorem violations
# TODO: Find correlations

# Your code here:
pass

---

**Congratulations!** You've mastered thermodynamic analysis of neural networks. You can now measure the physical costs of computation and verify fundamental thermodynamic laws.

Continue to **Notebook 15: Energy Cascades & Hamiltonian Decomposition** to understand the detailed energy flow structure.